# Historic data

In [ ]:
import sys
import os
import pandas as pd
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
print(project_root)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Importing scrappers
from src.scraper import download_omie_price
from src.scraper import download_omie_range_v2

In [ ]:
from datetime import datetime
today = datetime.today().strftime("%Y%m%d")
print(today)

In [ ]:
#df_historic = download_omie_range_v2("20000101", today)
# THERE ARE NO HISTORICAL DATA!!!

In [ ]:
# Here installation of OMIE data from 2020
#!pip install OMIEData

In [ ]:
import datetime as dt
from OMIEData.DataImport.omie_marginalprice_importer import OMIEMarginalPriceFileImporter

dateIni = dt.datetime(2019, 1, 1)
dateEnd = dt.datetime(2024, 12, 31)

df_historic = OMIEMarginalPriceFileImporter(
    date_ini=dateIni, 
    date_end=dateEnd
).read_to_dataframe(verbose=True)

print(df_historic.shape)
print(df_historic.head())

In [ ]:
df_historic.head(10)

In [ ]:
df_historic.shape

In [ ]:
# Filtrar solo precio España
df_price = df_historic[df_historic["CONCEPT"] == "PRICE_SP"].copy()

# De formato ancho a largo
hour_cols = [f"H{i}" for i in range(1, 25)]
df_long = df_price.melt(
    id_vars=["DATE"],
    value_vars=hour_cols,
    var_name="hour",
    value_name="price_es"
)

# Convertir hora a número
df_long["hour"] = df_long["hour"].str.replace("H", "").astype(int)

# Ordenar
df_long = df_long.sort_values(["DATE", "hour"]).reset_index(drop=True)

print(df_long.shape)
print(df_long.head())

In [ ]:
print(df_long["DATE"].min())
print(df_long["DATE"].max())

In [ ]:
df_long["DATE"] = pd.to_datetime(df_long["DATE"])
df_long["year"] = df_long["DATE"].dt.year
df_long["month"] = df_long["DATE"].dt.month
df_long["day"] = df_long["DATE"].dt.day
df_long["datetime"] = pd.to_datetime(df_long[["year", "month", "day"]].assign(hour=df_long["hour"] - 1))

# Renombrar DATE a date para consistencia
df_long = df_long.drop(columns="DATE")

print(df_long.shape)
print(df_long.head())

In [ ]:
import sqlite3
import os

base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
db_path = os.path.join(base_dir, "data", "electricity.db")
conn = sqlite3.connect(db_path)

df_long.to_sql("omie_prices_historic", conn, if_exists="replace", index=False)
print(f"Tabla creada con {len(df_long)} filas")

# Verificar
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

In [ ]:
# Guardemos los datos del historico
df_long.to_csv(os.path.join(base_dir, "data", "raw", "omie_historic_2019_2024.csv"), index=False)

In [ ]:
conn.close()

## 2. Open-Meteo Weather Data
Hourly weather data for three representative Spanish locations:
Madrid (demand proxy), Sevilla (solar proxy), Zaragoza (wind proxy)

In [ ]:
import importlib
import src.scraper
importlib.reload(src.scraper)
from src.scraper import download_openmeteo

In [ ]:
# Coordenadas de las tres ciudades
locations = [
    {"name": "Madrid",   "lat": 40.4168, "lon": -3.7038},
    {"name": "Sevilla",  "lat": 37.3891, "lon": -5.9845},
    {"name": "Zaragoza", "lat": 41.6488, "lon": -0.8891},
]

start_date = "2019-01-01"
end_date = "2024-12-31"

dfs_meteo = []

for loc in locations:
    print(f"Descargando {loc['name']}...")
    df_loc = download_openmeteo(
        lat=loc["lat"],
        lon=loc["lon"],
        start_date=start_date,
        end_date=end_date,
        location_name=loc["name"]
    )
    dfs_meteo.append(df_loc)
    print(f"OK: {len(df_loc)} filas")

df_meteo = pd.concat(dfs_meteo, ignore_index=True)
print(f"\nTotal: {df_meteo.shape}")
print(df_meteo.head())

In [ ]:
# Pivotar — una fila por hora, columnas por ciudad y variable
df_meteo_pivot = df_meteo.pivot_table(
    index="datetime",
    columns="location",
    values=["temperature", "solar_radiation", "wind_speed"]
)

# Aplanar nombres de columnas
df_meteo_pivot.columns = [f"{var}_{loc.lower()}" for var, loc in df_meteo_pivot.columns]
df_meteo_pivot = df_meteo_pivot.reset_index()

print(df_meteo_pivot.shape)
print(df_meteo_pivot.columns.tolist())

In [ ]:
import sqlite3
import os

base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
db_path = os.path.join(base_dir, "data", "electricity.db")
conn = sqlite3.connect(db_path)

df_meteo_pivot.to_sql("weather_Mad_Sev_Zar", conn, if_exists="replace", index=False)
print(f"Tabla weather creada con {df_meteo_pivot.shape[0]} filas y {df_meteo_pivot.shape[1]} columnas")

# Verificar tablas
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

In [ ]:
# pd.read_sql SOLO PARA CONSULTAS, NO MODIFICA LA BASE DE DATOS
#pd.read_sql("DROP TABLE IF EXISTS weather;", conn)

In [ ]:
conn.execute("DROP TABLE IF EXISTS weather;")
conn.commit()

In [ ]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)